# Systematic Review Automation — Google Colab
**LLMs in Qualitative Data Analysis (2023–2026)**

Run cells top to bottom. Cell 5 gives you a public URL — no accounts needed.

---
### Before you start — add your OpenAI API key as a Colab Secret
1. Click the **🔑 key icon** in the left sidebar → **+ Add new secret**
2. Name: `OPENAI_API_KEY` · Value: your `sk-...` key · Toggle **Notebook access** ON

Your key is encrypted in your Google account — never visible in output, never sent anywhere.


## Cell 1 — Clone repo + install dependencies

In [ ]:
import os, sys

REPO_URL = 'https://github.com/qbzdm2024/Zhihong.github.io.git'
BRANCH   = 'claude/systematic-review-automation-oC3BZ'
REPO_DIR = '/content/sysrev'
APP_DIR  = f'{REPO_DIR}/systematic-review'

if not os.path.exists(REPO_DIR):
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    print('Repo already present — pulling latest...')
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {APP_DIR}
print('Working directory:', os.getcwd())

In [ ]:
%%capture
!pip install \
    "fastapi==0.115.0" "uvicorn[standard]" \
    "pydantic>=2.7.0" "pydantic-settings>=2.3.0" \
    "openai>=1.50.0" "rispy>=0.9.0" \
    "python-multipart" "python-dotenv" "PyMuPDF" \
    "requests" "xmltodict"

In [ ]:
import fastapi, openai, pydantic, requests
print(f'fastapi {fastapi.__version__} | openai {openai.__version__} | pydantic {pydantic.__version__}')
for d in ['data/raw','data/deduped','data/screened','data/extracted','data/output','data/pdfs']:
    os.makedirs(d, exist_ok=True)
print('All packages installed. Data directories ready.')

## Cell 2 — Load API key from Colab Secrets

In [ ]:
from google.colab import userdata

try:
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
except Exception:
    raise RuntimeError(
        'OPENAI_API_KEY not found in Colab Secrets.\n'
        'Click the 🔑 key icon → Add new secret → Name: OPENAI_API_KEY'
    )

if not OPENAI_API_KEY or not OPENAI_API_KEY.startswith('sk-'):
    raise ValueError('Key must start with sk-')

with open('.env', 'w') as f:
    f.write(f'OPENAI_API_KEY={OPENAI_API_KEY}\n')
    f.write('MODEL_TITLE_SCREENING=gpt-4o-mini\n')
    f.write('MODEL_FULLTEXT_SCREENING=gpt-4o\n')
    f.write('MODEL_EXTRACTION=gpt-4o\n')
    f.write('MODEL_QA_ASSESSMENT=gpt-4o\n')
    f.write('MODEL_AGENT2_SCREENING=gpt-4o\n')
    f.write('MODEL_AGENT2_EXTRACTION=gpt-4o-mini\n')
    f.write('CONFIDENCE_THRESHOLD=0.80\n')

print(f'✓ API key loaded (sk-...{OPENAI_API_KEY[-4:]})')
print('✓ .env written — setup wizard will be skipped')

## Cell 3a — Automated database search (free, no login required)

Searches **PubMed, arXiv, OpenAlex, and Semantic Scholar** automatically.
All results are saved to `data/raw/` ready for import.

These four sources are free and open — **no institutional access needed.**
For Scopus, Web of Science, PsycINFO, ACM, and IEEE, see Cell 3b below.

In [ ]:
import requests, json, time, xml.etree.ElementTree as ET
from urllib.parse import quote_plus
from datetime import datetime

os.makedirs('data/raw', exist_ok=True)

# ── Search configuration ────────────────────────────────
DATE_FROM = '2023'
DATE_TO   = '2026'
MAX_RESULTS_PER_SOURCE = 500   # increase if you want more

# ── Core search terms (from refined protocol) ───────────
LLM_TERMS = [
    'large language model', 'large language models', 'LLM', 'LLMs',
    'generative AI', 'generative artificial intelligence',
    'GPT-4', 'GPT-3', 'GPT-3.5', 'ChatGPT', 'GPT',
    'Claude', 'Llama', 'Llama 2', 'Llama 3',
    'Gemini', 'Mistral', 'PaLM', 'foundation model',
]
QDA_TERMS = [
    'qualitative analysis', 'qualitative research', 'qualitative data analysis',
    'thematic analysis', 'content analysis', 'grounded theory',
    'framework analysis', 'narrative analysis', 'discourse analysis',
    'inductive coding', 'deductive coding', 'open coding',
    'codebook', 'qualitative coding', 'interpretive phenomenological',
]

print(f'Search scope: {DATE_FROM}–{DATE_TO} | max {MAX_RESULTS_PER_SOURCE} per source')
print(f'LLM terms: {len(LLM_TERMS)} | QDA terms: {len(QDA_TERMS)}')

In [ ]:
# ════════════════════════════════════════════════════════
# PubMed  (NCBI E-utilities — completely free, no key)
# ════════════════════════════════════════════════════════
print('Searching PubMed...')

EUTILS = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils'

llm_q  = ' OR '.join(f'"{t}"[Title/Abstract]' for t in LLM_TERMS)
qda_q  = ' OR '.join(f'"{t}"[Title/Abstract]' for t in QDA_TERMS)
pubmed_query = f'({llm_q}) AND ({qda_q}) AND ("{DATE_FROM}/01/01"[Date - Publication] : "{DATE_TO}/12/31"[Date - Publication])'

# Step 1: get IDs
search_r = requests.get(f'{EUTILS}/esearch.fcgi', params={
    'db': 'pubmed', 'term': pubmed_query,
    'retmax': MAX_RESULTS_PER_SOURCE, 'retmode': 'json',
    'usehistory': 'y',
}, timeout=30)
search_data = search_r.json()
count  = int(search_data['esearchresult']['count'])
webenv = search_data['esearchresult'].get('webenv', '')
qkey   = search_data['esearchresult'].get('querykey', '')
ids    = search_data['esearchresult']['idlist']
print(f'  Total matching: {count} | Fetching: {len(ids)}')

# Step 2: fetch XML records
if ids:
    time.sleep(0.4)  # NCBI rate limit: 3 req/s
    fetch_r = requests.get(f'{EUTILS}/efetch.fcgi', params={
        'db': 'pubmed', 'query_key': qkey, 'WebEnv': webenv,
        'rettype': 'xml', 'retmode': 'xml',
        'retmax': MAX_RESULTS_PER_SOURCE,
    }, timeout=60)
    out_path = 'data/raw/pubmed_results.xml'
    with open(out_path, 'wb') as f:
        f.write(fetch_r.content)
    print(f'  ✓ Saved {len(ids)} records → {out_path}')
else:
    print('  No PubMed results found.')

time.sleep(0.4)

In [ ]:
# ════════════════════════════════════════════════════════
# arXiv  (free API, no key — cs.AI, cs.CL, cs.HC)
# ════════════════════════════════════════════════════════
print('Searching arXiv...')

import xml.etree.ElementTree as ET

llm_arxiv = ' OR '.join(f'abs:"{t}"' for t in LLM_TERMS[:8])  # keep query short
qda_arxiv = ' OR '.join(f'abs:"{t}"' for t in QDA_TERMS[:8])
arxiv_query = f'({llm_arxiv}) AND ({qda_arxiv})'

arxiv_records = []
batch_size = 100
start = 0
import uuid

while start < MAX_RESULTS_PER_SOURCE:
    r = requests.get('http://export.arxiv.org/api/query', params={
        'search_query': arxiv_query,
        'start': start,
        'max_results': batch_size,
        'sortBy': 'submittedDate',
        'sortOrder': 'descending',
    }, timeout=30)

    ns = {'atom': 'http://www.w3.org/2005/Atom'}
    root = ET.fromstring(r.content)
    entries = root.findall('atom:entry', ns)
    if not entries:
        break

    for entry in entries:
        published = entry.findtext('atom:published', '', ns)[:4]
        if published < DATE_FROM or published > DATE_TO:
            continue
        arxiv_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'arXiv',
            'title': (entry.findtext('atom:title', '', ns) or '').strip().replace('\n', ' '),
            'authors': '; '.join(
                (a.findtext('atom:name', '', ns) or '')
                for a in entry.findall('atom:author', ns)
            ),
            'year': int(published) if published.isdigit() else None,
            'journal_venue': 'arXiv preprint',
            'abstract': (entry.findtext('atom:summary', '', ns) or '').strip().replace('\n', ' '),
            'doi': '',
            'url': entry.findtext('atom:id', '', ns),
        })

    start += batch_size
    if len(entries) < batch_size:
        break
    time.sleep(3)  # arXiv asks for 3s between requests

if arxiv_records:
    out_path = 'data/raw/arxiv_results.json'
    with open(out_path, 'w') as f:
        json.dump(arxiv_records, f, indent=2)
    print(f'  ✓ Saved {len(arxiv_records)} records → {out_path}')
else:
    print('  No arXiv results in date range.')

In [ ]:
# ════════════════════════════════════════════════════════
# OpenAlex  (free, open — covers Scopus/WoS/PubMed)
# https://openalex.org — no API key needed for <100k req/day
# ════════════════════════════════════════════════════════
print('Searching OpenAlex...')

# Build search query — OpenAlex uses simple keyword search
llm_kw  = ' OR '.join(f'"{t}"' for t in [
    'large language model', 'ChatGPT', 'GPT-4', 'LLM', 'Llama', 'Gemini', 'generative AI'
])
qda_kw  = ' OR '.join(f'"{t}"' for t in [
    'qualitative analysis', 'thematic analysis', 'content analysis',
    'grounded theory', 'qualitative research', 'qualitative coding',
])

openalex_records = []
page = 1
per_page = 100
EMAIL = 'sysrev@example.com'  # polite pool (faster responses)

while len(openalex_records) < MAX_RESULTS_PER_SOURCE:
    r = requests.get('https://api.openalex.org/works', params={
        'search': f'({llm_kw}) AND ({qda_kw})',
        'filter': f'publication_year:{DATE_FROM}-{DATE_TO},language:en',
        'per-page': per_page,
        'page': page,
        'mailto': EMAIL,
    }, timeout=30)

    if r.status_code != 200:
        print(f'  OpenAlex error {r.status_code}: {r.text[:200]}')
        break

    data = r.json()
    results = data.get('results', [])
    if not results:
        break

    for w in results:
        authors = '; '.join(
            a.get('author', {}).get('display_name', '')
            for a in (w.get('authorships') or [])[:6]
        )
        abstract = ''
        if w.get('abstract_inverted_index'):
            # Reconstruct abstract from inverted index
            idx = w['abstract_inverted_index']
            words = {pos: word for word, positions in idx.items() for pos in positions}
            abstract = ' '.join(words[i] for i in sorted(words))

        doi = w.get('doi') or ''
        if doi.startswith('https://doi.org/'):
            doi = doi[len('https://doi.org/'):]

        venue = ''
        if w.get('primary_location') and w['primary_location'].get('source'):
            venue = w['primary_location']['source'].get('display_name', '')

        openalex_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'OpenAlex',
            'title': w.get('title') or '',
            'authors': authors,
            'year': w.get('publication_year'),
            'journal_venue': venue,
            'abstract': abstract,
            'doi': doi,
            'url': w.get('id') or '',
        })

    total = data.get('meta', {}).get('count', 0)
    print(f'  Page {page}: {len(results)} results (total available: {total})')

    if page * per_page >= min(total, MAX_RESULTS_PER_SOURCE):
        break
    page += 1
    time.sleep(1)

if openalex_records:
    out_path = 'data/raw/openalex_results.json'
    with open(out_path, 'w') as f:
        json.dump(openalex_records, f, indent=2)
    print(f'  ✓ Saved {len(openalex_records)} records → {out_path}')
else:
    print('  No OpenAlex results.')

In [ ]:
# ════════════════════════════════════════════════════════
# Semantic Scholar  (free API — strong CS/AI coverage)
# https://api.semanticscholar.org
# ════════════════════════════════════════════════════════
print('Searching Semantic Scholar...')

s2_records = []
offset = 0
limit = 100
query = 'large language model qualitative analysis thematic analysis coding'

while len(s2_records) < MAX_RESULTS_PER_SOURCE:
    r = requests.get('https://api.semanticscholar.org/graph/v1/paper/search', params={
        'query': query,
        'fields': 'title,authors,year,abstract,externalIds,venue,publicationDate',
        'limit': limit,
        'offset': offset,
    }, timeout=30)

    if r.status_code == 429:
        print('  Rate limited — waiting 10s...')
        time.sleep(10)
        continue

    if r.status_code != 200:
        print(f'  S2 error {r.status_code}')
        break

    data = r.json()
    papers = data.get('data', [])
    if not papers:
        break

    for p in papers:
        year = p.get('year')
        if year and (year < int(DATE_FROM) or year > int(DATE_TO)):
            continue

        doi = (p.get('externalIds') or {}).get('DOI', '')
        authors = '; '.join(
            a.get('name', '') for a in (p.get('authors') or [])[:6]
        )
        s2_records.append({
            'record_id': str(uuid.uuid4()),
            'source_db': 'SemanticScholar',
            'title': p.get('title') or '',
            'authors': authors,
            'year': year,
            'journal_venue': p.get('venue') or '',
            'abstract': p.get('abstract') or '',
            'doi': doi,
            'url': f'https://www.semanticscholar.org/paper/{p["paperId"]}',
        })

    total = data.get('total', 0)
    print(f'  Offset {offset}: got {len(papers)} (total: {total})')

    if offset + limit >= min(total, MAX_RESULTS_PER_SOURCE):
        break
    offset += limit
    time.sleep(1.5)

if s2_records:
    out_path = 'data/raw/semanticscholar_results.json'
    with open(out_path, 'w') as f:
        json.dump(s2_records, f, indent=2)
    print(f'  ✓ Saved {len(s2_records)} records → {out_path}')

print('\n── Automated search summary ──────────────────────')
for fname in ['pubmed_results.xml','arxiv_results.json',
              'openalex_results.json','semanticscholar_results.json']:
    p = f'data/raw/{fname}'
    if os.path.exists(p):
        size = os.path.getsize(p)
        print(f'  ✓ {fname:45s} {size:>8,} bytes')
    else:
        print(f'  ✗ {fname} — not generated')

## Cell 3b — Manual exports (institutional databases)

The databases below require university/library login. Run these searches yourself,
then upload the exported files using the upload cell below.

**Copy-paste this exact search string into each database:**

```
("large language model" OR "large language models" OR LLM OR LLMs
 OR "generative AI" OR GPT-4 OR GPT-3 OR ChatGPT OR Claude OR Llama OR Gemini
 OR "foundation model")
AND
("qualitative analysis" OR "qualitative research" OR "qualitative data analysis"
 OR "thematic analysis" OR "content analysis" OR "grounded theory"
 OR "framework analysis" OR "narrative analysis" OR "discourse analysis"
 OR "inductive coding" OR "deductive coding" OR codebook)
```

---

### Scopus
1. Go to [scopus.com](https://www.scopus.com) (requires institutional login)
2. Click **Advanced search** → paste the query above
3. Add filter: **Date range 2023–2026** · **Language: English**
4. Select all results → **Export** → **RIS format** → include Abstract + Keywords
5. Save file as `scopus.ris`

### Web of Science
1. Go to [webofscience.com](https://www.webofscience.com) → **Advanced Search**
2. Paste query, set **Timespan: 2023–2026**, **Language: English**
3. Select all → **Export** → **Other file formats** → **Plain Text** or **RIS** → include all fields
4. Save as `wos.ris`

### PsycINFO (via EBSCOhost)
1. Access via your library → EBSCOhost → PsycINFO database
2. **Advanced Search** → paste query
3. Limiter: **Publication Date 2023–2026** · **Language: English** · **Peer Reviewed**
4. Select all → **Export** → **Direct Export to RIS**
5. Save as `psycinfo.ris`

### ACM Digital Library
1. Go to [dl.acm.org](https://dl.acm.org) → **Advanced Search**
2. Set fields: Abstract/Title contains LLM terms AND QDA terms
3. Filter: **Published 2023–2026**
4. Export → **BibTeX** or **CSV**
5. Save as `acm.bib` or `acm.csv`

### IEEE Xplore
1. Go to [ieeexplore.ieee.org](https://ieeexplore.ieee.org) → **Advanced Search**
2. Paste query, set **Year: 2023–2026**
3. Export → **CSV** → include Abstract
4. Save as `ieee.csv`


### Upload manual export files

In [ ]:
from google.colab import files as colab_files

SUPPORTED = ('.ris', '.csv', '.bib', '.json', '.xml')
print('Select your exported files (Scopus .ris, WoS .ris, PsycINFO .ris, ACM .bib, IEEE .csv):')
print('Press Cancel if you have no files to upload right now.\n')

try:
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        if any(fname.lower().endswith(ext) for ext in SUPPORTED):
            dest = f'data/raw/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  ✓ {fname} ({len(content):,} bytes) → data/raw/')
        else:
            print(f'  ✗ Skipped (unsupported): {fname}')
except Exception:
    print('No files uploaded.')

print('\nAll files in data/raw/:')
!ls -lh data/raw/

## Cell 4 — (Optional) Upload PDFs for full-text screening

Name each file `<record_id>.pdf`. You can also do this later via the UI.

In [ ]:
from google.colab import files as colab_files
print('Select PDF files (Cancel to skip):')
try:
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        if fname.lower().endswith('.pdf'):
            dest = f'data/pdfs/{fname}'
            with open(dest, 'wb') as f:
                f.write(content)
            print(f'  ✓ {fname}')
        else:
            print(f'  ✗ Skipped (not PDF): {fname}')
except Exception:
    print('No PDFs uploaded — add them later via the UI.')

## Cell 5 — Start server + open public URL

Uses Cloudflare Tunnel — no account, no token, free.

In [ ]:
import subprocess, time, re, sys, os, urllib.request
from IPython.display import display, HTML

APP_DIR = '/content/sysrev/systematic-review'
PORT = 8000

os.system(f'fuser -k {PORT}/tcp 2>/dev/null; pkill cloudflared 2>/dev/null; true')
time.sleep(1)

if not os.path.exists('/content/cloudflared'):
    print('Downloading cloudflared...')
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 \
         -O /content/cloudflared && chmod +x /content/cloudflared

server = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'api.main:app',
     '--host', '0.0.0.0', '--port', str(PORT), '--log-level', 'warning'],
    cwd=APP_DIR, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)

print('Starting server', end='')
for _ in range(30):
    time.sleep(1)
    try:
        urllib.request.urlopen(f'http://localhost:{PORT}/api/pipeline/status', timeout=2)
        print(' ready.')
        break
    except Exception:
        print('.', end='', flush=True)
else:
    print('\nERROR: Server did not start. Re-run this cell.')

tunnel = subprocess.Popen(
    ['/content/cloudflared', 'tunnel', '--url', f'http://localhost:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

public_url = None
print('Opening Cloudflare tunnel', end='')
for _ in range(40):
    line = tunnel.stdout.readline().decode('utf-8', errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print(' done.')
        break
    print('.', end='', flush=True)

if public_url:
    app_url = f'{public_url}/app'
    print(f'\n  UI:  {app_url}')
    print(f'  API: {public_url}/docs')
    display(HTML(f'''
    <div style="margin:12px 0;padding:18px;background:#0f1117;border:2px solid #6c8eff;
      border-radius:10px;font-family:monospace;">
      <div style="color:#8890aa;font-size:11px;margin-bottom:6px;">OPEN IN YOUR BROWSER</div>
      <a href="{app_url}" target="_blank" rel="noopener"
        style="color:#6c8eff;font-size:18px;font-weight:bold;text-decoration:none;">{app_url}</a>
      <div style="color:#4caf81;font-size:12px;margin-top:10px;">
        ✓ API key from Colab Secrets &nbsp;|&nbsp; ✓ No account needed &nbsp;|&nbsp; ✓ HTTPS
      </div>
    </div>
    '''))
else:
    print('\nERROR: Could not get tunnel URL. Re-run this cell.')

## Cell 6 — Run pipeline stages

Run from here, or use the **Run Pipeline** panel in the UI.

In [ ]:
import requests

PORT = 8000   # defined here so this cell works independently
BASE = f'http://localhost:{PORT}/api'

def run_stage(stage, limit=None):
    r = requests.post(f'{BASE}/pipeline/run', json={'stage': stage, 'limit': limit})
    r.raise_for_status()
    print(f'Started: {stage}', r.json())

def status():
    counts = requests.get(f'{BASE}/pipeline/status').json()['prisma_counts']
    print('\n── Pipeline Status ───────────────────────────')
    for k, v in counts.items():
        mark = ' ⚠' if k == 'needs_human_verification' and v > 0 else ''
        print(f'  {k:42s} {v}{mark}')
    print()

def uncertain():
    data = requests.get(f'{BASE}/records/uncertain/list').json()
    print(f'\nNeeds Human Verification: {data["count"]} records')
    for r in data['records'][:10]:
        print(f'  [{r["record_id"][:8]}] {r["title"][:70]}')

def fulltext_needed():
    data = requests.get(f'{BASE}/records/fulltext-needed/list').json()
    print(f'\nFull Text Needed: {data["count"]} papers')
    for r in data['records'][:10]:
        print(f'  {r["doi"] or "no-doi":40s}  {r["title"][:50]}')

print('Helpers ready: run_stage() | status() | uncertain() | fulltext_needed()')

In [ ]:
# Uncomment and run one at a time. Check status() between stages.

# run_stage('import')          # load all files from data/raw/
# status()

# run_stage('dedup')           # remove duplicates
# status()

# run_stage('title_screening') # two-agent title/abstract screening
# status()
# uncertain()                  # ← review these in the UI before continuing

# run_stage('fulltext_screening')
# fulltext_needed()            # ← upload those PDFs, then re-run

# run_stage('extraction')
# status()

## Cell 7 — Save to Google Drive (run before closing)

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)
DRIVE = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP   = '/content/sysrev/systematic-review'
os.makedirs(DRIVE, exist_ok=True)

for src, label in [
    (f'{APP}/data/pipeline_state.jsonl', 'pipeline_state.jsonl'),
    (f'{APP}/data/output', 'output/'),
    (f'{APP}/data/raw',    'raw/'),
    (f'{APP}/data/pdfs',   'pdfs/'),
]:
    if not os.path.exists(src):
        continue
    if os.path.isfile(src):
        shutil.copy(src, f'{DRIVE}/{label}')
    else:
        if os.listdir(src):
            shutil.copytree(src, f'{DRIVE}/{label.rstrip("/")}', dirs_exist_ok=True)
    print(f'  ✓ Saved: {label}')

print(f'\nSaved to Google Drive: {DRIVE}')

## Cell 8 — Restore after session reset

Run: **Cell 1 → Cell 2 → this cell → Cell 5** to resume.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive', force_remount=False)
DRIVE = '/content/drive/MyDrive/SysRev_LLMs_QDA'
APP   = '/content/sysrev/systematic-review'

if not os.path.exists(DRIVE):
    print('No saved session in Drive — starting fresh.')
else:
    restored = []
    for src_name, dest_path, is_file in [
        ('pipeline_state.jsonl', f'{APP}/data/pipeline_state.jsonl', True),
        ('output',               f'{APP}/data/output',               False),
        ('raw',                  f'{APP}/data/raw',                  False),
        ('pdfs',                 f'{APP}/data/pdfs',                 False),
    ]:
        src = f'{DRIVE}/{src_name}'
        if not os.path.exists(src):
            continue
        os.makedirs(os.path.dirname(dest_path), exist_ok=True)
        if is_file:
            shutil.copy(src, dest_path)
        else:
            shutil.copytree(src, dest_path, dirs_exist_ok=True)
        restored.append(src_name)

    print(f'Restored: {" | ".join(restored) or "nothing found"}')
    print('Now run Cell 5 to restart the server.')

## Cell 9 — Stop the server

In [ ]:
try:
    tunnel.terminate()
    server.terminate()
except Exception:
    pass
os.system('pkill cloudflared 2>/dev/null; true')
print('Stopped.')

---
## Summary

| Database | How searched | Format saved |
|----------|-------------|-------------|
| **PubMed** | Cell 3a (auto, free API) | `pubmed_results.xml` |
| **arXiv** | Cell 3a (auto, free API) | `arxiv_results.json` |
| **OpenAlex** | Cell 3a (auto, free API — covers WoS/Scopus) | `openalex_results.json` |
| **Semantic Scholar** | Cell 3a (auto, free API) | `semanticscholar_results.json` |
| **Scopus** | Cell 3b (manual export, needs login) | `scopus.ris` |
| **Web of Science** | Cell 3b (manual export, needs login) | `wos.ris` |
| **PsycINFO** | Cell 3b (manual export, needs login) | `psycinfo.ris` |
| **ACM DL** | Cell 3b (manual export, needs login) | `acm.bib` |
| **IEEE Xplore** | Cell 3b (manual export, needs login) | `ieee.csv` |

All files land in `data/raw/`. The import stage handles deduplication across sources.
